# **Kết nối Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Setup**

## **Tải thư viện (nếu chưa có)**

In [3]:
!pip -q install transformers datasets seqeval accelerate underthesea

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.4 MB/s eta 0:00:00


## **Import thư viện & cấu hình**

In [ ]:
# Standard libs
import os
import json
import random
import unicodedata
from typing import Any, Dict, List, Tuple

# Numerical / Data
import numpy as np
import pandas as pd
import torch

# HuggingFace Datasets
from datasets import Dataset, DatasetDict, Features, Sequence, Value

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed,
)

# Metrics (seqeval)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Vietnamese word segmentation (optional: PhoBERT path)
from underthesea import word_tokenize

## **Các hàm tiện ích**

### Tiện ích xử lý JSONL & Unicode

- **`nfc(text)`**: Chuẩn hoá chuỗi Unicode về dạng **NFC** (quan trọng với tiếng Việt).
- **`load_jsonl(path)`**: Đọc file `.jsonl`, mỗi dòng là một JSON object; báo lỗi chi tiết nếu dòng không hợp lệ.
- **`save_jsonl(rows, path)`**: Ghi danh sách object ra file `.jsonl`, giữ nguyên Unicode.
- **`split_jsonl(...)`**: Trộn ngẫu nhiên và chia dữ liệu JSONL thành **train/valid/test** theo tỉ lệ chỉ định (cố định seed).
- **`read_jsonl_and_rename_label(path)`**: Đọc `.jsonl` và đổi khóa **`label` → `labels`** để đồng bộ schema.


In [5]:
def nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Invalid JSON at {path}:{ln}\nLine: {line}\nError: {e}")
    return rows

def save_jsonl(rows: List[Dict[str, Any]], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def split_jsonl(
    input_path: str,
    out_train: str,
    out_valid: str,
    out_test: str,
    train_ratio: float = 0.8,
    valid_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,
) -> None:
    assert abs(train_ratio + valid_ratio + test_ratio - 1.0) < 1e-9
    rows = load_jsonl(input_path)
    rng = random.Random(seed)
    rng.shuffle(rows)

    n = len(rows)
    n_train = int(n * train_ratio)
    n_valid = int(n * valid_ratio)

    train_rows = rows[:n_train]
    valid_rows = rows[n_train:n_train + n_valid]
    test_rows  = rows[n_train + n_valid:]

    save_jsonl(train_rows, out_train)
    save_jsonl(valid_rows, out_valid)
    save_jsonl(test_rows, out_test)
    print(f"Split: train={len(train_rows)}, valid={len(valid_rows)}, test={len(test_rows)}")

def read_jsonl_and_rename_label(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)

            # đổi label -> labels (nếu cần)
            if "label" in r and "labels" not in r:
                r["labels"] = r["label"]

            rows.append(r)
    return rows

### Kiểm tra & làm sạch dữ liệu gán nhãn (span-based)

- **`_check_no_overlap(spans)`**: Sắp xếp và phát hiện các span **chồng lấn**; báo lỗi nếu có overlap.
- **`validate_rows(rows, ...)`**:  
  - Kiểm tra bắt buộc có **`text`**, chuẩn hoá Unicode **NFC**.  
  - Xác thực **labels = [start, end, label]** (kiểu dữ liệu, phạm vi, độ dài).  
  - Kiểm tra **out-of-range**, **span rỗng**, **span chỉ toàn whitespace** (tuỳ `strict`).  
  - Phát hiện **overlap span**.  
  - Có thể **loại bỏ dòng lỗi** (`drop_invalid`) và **log chi tiết** (`verbose`).  
- **Kết quả**: Trả về danh sách dòng hợp lệ với schema chuẩn `{text, labels}`.

In [ ]:
def _check_no_overlap(spans: List[Tuple[int, int, str]]) -> None:
    spans_sorted = sorted(spans, key=lambda x: (x[0], x[1]))
    for i in range(1, len(spans_sorted)):
        ps, pe, _ = spans_sorted[i-1]
        cs, ce, _ = spans_sorted[i]
        if cs < pe:
            raise ValueError(
                f"Overlapping spans detected: prev={spans_sorted[i-1]} cur={spans_sorted[i]}"
            )

def validate_rows(
    rows: List[Dict[str, Any]],
    strict: bool = True,
    drop_invalid: bool = False,
    verbose: bool = True,
) -> List[Dict[str, Any]]:
    out = []
    removed = 0

    for idx, r in enumerate(rows):
        try:
            if "text" not in r:
                raise ValueError(f"Row {idx} missing 'text'. Row={r}")

            text = nfc(r["text"])
            labels = r.get("labels", []) or []

            spans: List[Tuple[int, int, str]] = []
            for item in labels:
                if not (isinstance(item, (list, tuple)) and len(item) == 3):
                    raise ValueError(f"Row {idx} invalid label item: {item}")

                s, e, lab = item
                if not (isinstance(s, int) and isinstance(e, int) and isinstance(lab, str)):
                    raise ValueError(f"Row {idx} label types invalid: {item}")
                if s < 0 or e < 0 or s > e:
                    raise ValueError(f"Row {idx} label offsets invalid: {item}")

                if e > len(text):
                    msg = f"Row {idx} label end out of range: {item}, len(text)={len(text)}"
                    if strict:
                        raise ValueError(msg)
                    e = min(e, len(text))

                if strict and (e - s) == 0:
                    raise ValueError(f"Row {idx} empty span: {item}")
                if strict and text[s:e].strip() == "":
                    raise ValueError(
                        f"Row {idx} span is only whitespace: {item} -> '{text[s:e]}'"
                    )

                spans.append((s, e, lab))

            _check_no_overlap(spans)

            out.append({
                "text": text,
                "labels": [[s, e, lab] for s, e, lab in spans]
            })

        except Exception as err:
            if strict and not drop_invalid:
                raise
            removed += 1
            if verbose:
                print(f"[DROP] Row {idx}: {err}")

    if verbose:
        print(
            f"Validation done | total={len(rows)} | kept={len(out)} | removed={removed}"
        )

    return out

### Xây dựng hệ nhãn BIO cho NER (token-level)

- **`build_bio_label_maps(rows_all)`**:  
  Duyệt toàn bộ dữ liệu để thu thập các **loại thực thể** (entity types) xuất hiện trong `labels`.

- **Cách hoạt động**:
  - Gom tất cả nhãn thực thể (vd: `PERSON`, `ORG`) từ các span `[start, end, label]`.
  - Tạo hệ nhãn **BIO** gồm:
    - `O`: không thuộc thực thể
    - `B-<ENTITY>`: token bắt đầu thực thể
    - `I-<ENTITY>`: token nằm trong thực thể
  - Sắp xếp nhãn thực thể để đảm bảo **thứ tự ổn định, tái lập được**.

- **Output**:
  - `labels`: danh sách nhãn BIO (string)
  - `label2id`: ánh xạ nhãn → chỉ số (dùng khi train)
  - `id2label`: ánh xạ chỉ số → nhãn (dùng khi inference)

- **Mục đích**:  
  Chuẩn hoá không gian nhãn cho bài toán **Named Entity Recognition** ở mức token, phục vụ huấn luyện và suy luận mô hình.

In [7]:
def build_bio_label_maps(rows_all: List[Dict[str, Any]]) -> Tuple[List[str], Dict[str, int], Dict[int, str]]:
    entity_types = set()
    for r in rows_all:
        for s, e, lab in r.get("labels", []):
            entity_types.add(lab)

    labels = ["O"]
    for ent in sorted(entity_types):
        labels.append(f"B-{ent}")
        labels.append(f"I-{ent}")

    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    return labels, label2id, id2label

### Chuyển dữ liệu span-level sang HuggingFace Dataset

- **`FEATURES`**: Định nghĩa schema chuẩn cho HuggingFace Dataset gồm:
  - `text`: văn bản gốc
  - `spans_start`: danh sách vị trí bắt đầu span (char-level)
  - `spans_end`: danh sách vị trí kết thúc span
  - `spans_label`: danh sách nhãn thực thể tương ứng

- **`to_dataset(rows)`**:
  - Nhận dữ liệu đầu vào dạng `{text, labels=[[start, end, label], ...]}`.
  - Tách mỗi span thành **ba mảng song song** (`start`, `end`, `label`) theo đúng thứ tự xuất hiện.
  - Chuyển danh sách record sang **HuggingFace `Dataset`** với schema đã khai báo.

- **Mục đích**:  
  Chuẩn hoá dữ liệu NER ở mức **character-span** để dễ dàng tích hợp vào pipeline tokenization, align span → token và huấn luyện mô hình trên HuggingFace.

In [ ]:
FEATURES = Features({
    "text": Value("string"),
    "spans_start": Sequence(Value("int32")),
    "spans_end": Sequence(Value("int32")),
    "spans_label": Sequence(Value("string")),
})

def to_dataset(rows):
    items = []
    for r in rows:
        text = r["text"]
        raw = r.get("labels", []) or []   # raw = [[s,e,label], ...]

        starts, ends, labs = [], [], []
        for it in raw:
            s, e, lab = it
            starts.append(int(s))
            ends.append(int(e))
            labs.append(str(lab))

        items.append({
            "text": text,
            "spans_start": starts,
            "spans_end": ends,
            "spans_label": labs
        })

    return Dataset.from_list(items, features=FEATURES)

### Căn chỉnh nhãn NER từ span (char-level) → token labels (BIO)

- **`build_char_mask(text, spans, prefer)`**: Tạo “mặt nạ ký tự” dài bằng `len(text)`, mỗi vị trí lưu label thực thể hoặc `None`.
  - `prefer="first"`: giữ span xuất hiện trước nếu overlap
  - `prefer="last"`: span sau ghi đè span trước

#### Nhánh 1 — Fast tokenizer (có `offset_mapping`)
- **`align_spans_to_token_labels(tokenizer, label2id, examples)`**:
  - Tokenize batch và lấy **offset (start,end)** của từng token trên chuỗi gốc.
  - Với mỗi token:
    - Nếu `(st==en)` → gán `-100` (bỏ qua, thường là special token)
    - Tìm label bằng **ký tự đầu tiên trong token** thuộc entity (dò trên `mask`)
    - Quyết định **B/I** bằng `begin_set` (token bắt đầu đúng tại `span.start` → `B-`, ngược lại → `I-`)
  - Trả về `input_ids`, `attention_mask`, `labels` (đã bỏ `offset_mapping`)

#### Nhánh 2 — PhoBERT (không có offsets)
- **`_tokenize_vi_words(text)`**: Word segmentation (ưu tiên `underthesea`, fallback split theo whitespace).
- **`viet_words_with_offsets_robust(text)`**: Tạo offset từ token đã tách từ bằng **pointer scanning** để map token → vị trí (start,end) trong text.
- **`word_bio_tags_robust(text, spans)`**:
  - Dùng `mask` để gán label cho từng word theo ký tự đầu tiên thuộc entity trong word
  - Quyết định B/I theo `begin_set` (word_start == span.start cùng label)
- **`encode_words_and_labels(tokenizer, tokens, word_tags, label2id)`**:
  - Tokenize từng word thành subword
  - Gán nhãn:
    - subword đầu giữ `B-/I-`
    - các subword sau: nếu `B-` thì chuyển thành `I-`
  - Thêm special tokens và gán `-100` ở đầu/cuối
- **`align_spans_to_token_labels_phobert(...)`**: Pipeline hoàn chỉnh PhoBERT: spans → word BIO → subword labels.

#### Kiểm tra
- **`sanity_check_alignment(batch_enc)`**: Đảm bảo `len(input_ids) == len(labels) == len(attention_mask)` cho một vài mẫu.
- **`has_offset_mapping(tokenizer)`**: Kiểm tra tokenizer có phải fast tokenizer (có offsets) hay không.

In [ ]:
def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s or "")

def build_char_mask(text: str, spans: List[Tuple[int, int, str]], *, prefer="first"):
    """
    text: normalized NFC
    spans: list[(start,end,label)] char-level
    prefer:
      - "first": giữ label span đến trước nếu overlap
      - "last": span sau ghi đè span trước nếu overlap
    """
    mask = [None] * len(text)

    # sort spans by start, then by length desc (optional)
    spans_sorted = sorted(
        [(int(s), int(e), lab) for s, e, lab in spans],
        key=lambda x: (x[0], -(x[1]-x[0]))
    )

    for s, e, lab in spans_sorted:
        s = max(0, min(s, len(text)))
        e = max(0, min(e, len(text)))
        if e <= s:
            continue
        for i in range(s, e):
            if prefer == "first":
                if mask[i] is None:
                    mask[i] = lab
            else:  # "last"
                mask[i] = lab
    return mask

# FAST TOKENIZER WITH OFFSETS 
def align_spans_to_token_labels(tokenizer, label2id: Dict[str, int], examples: Dict[str, Any]):
    """
    Output keys: input_ids, attention_mask, labels
    - Dùng offset_mapping: token nào đè lên entity -> gán label theo ký tự đầu tiên của entity trong token.
    - Quyết định B/I dựa theo việc token.start có đúng là một span.start cùng label hay không.
    """
    texts = [nfc(t) for t in examples["text"]]
    starts_batch = examples["spans_start"]
    ends_batch   = examples["spans_end"]
    labs_batch   = examples["spans_label"]

    enc = tokenizer(
        texts,
        truncation=True,
        return_offsets_mapping=True,
    )

    all_labels = []
    for text, starts, ends, labs, offsets in zip(texts, starts_batch, ends_batch, labs_batch, enc["offset_mapping"]):
        spans = [(int(s), int(e), lab) for s, e, lab in zip(starts, ends, labs)]
        mask = build_char_mask(text, spans, prefer="first")

        # set of (start,label) để check begin
        begin_set = {(max(0, min(int(s), len(text))), lab) for (s, e, lab) in spans}

        token_labels = []
        for (st, en) in offsets:
            if st == en:
                token_labels.append(-100)
                continue

            st = max(0, min(int(st), len(text)))
            en = max(0, min(int(en), len(text)))

            # tìm label theo ký tự đầu tiên trong token mà thuộc entity
            lab = None
            for i in range(st, min(en, len(mask))):
                if mask[i] is not None:
                    lab = mask[i]
                    break

            if lab is None:
                token_labels.append(label2id["O"])
            else:
                tag = f"B-{lab}" if ((st, lab) in begin_set) else f"I-{lab}"
                token_labels.append(label2id[tag])

        all_labels.append(token_labels)

    enc.pop("offset_mapping")
    enc["labels"] = all_labels
    return enc

# PHOBERT (NO OFFSETS) - WORD ALIGNMENT ROBUST
def _tokenize_vi_words(text: str):
    """
    Prefer underthesea if available. Fallback: whitespace split.
    Return list of tokens (strings) as displayed in segmented text.
    """
    text = nfc(text)
    try:
        from underthesea import word_tokenize
        seg = word_tokenize(text, format="text")  # "Nhân_phận : ..."
        return seg.split()
    except Exception:
        return text.split()


def _scan_match_token(text: str, token: str, pos: int):
    """
    Map a segmented token -> (start,end) in original text using pointer scanning.
    - token may contain '_' meaning space in original.
    - Skip whitespace in original text when needed.
    Strategy:
      Convert token to raw = token with '_' -> ' '.
      Then scan forward in text from pos to find the next occurrence *sequentially*,
      but in a controlled way that avoids global find drift:
        - move pos until first char matches,
        - then check full raw match.
    Returns (start,end,new_pos) or (None,None,pos) if cannot match.
    """
    raw = token.replace("_", " ")
    n = len(text)

    # skip leading spaces
    while pos < n and text[pos].isspace():
        pos += 1

    if not raw:
        return None, None, pos

    # attempt sequential match
    # try a limited scan forward
    first = raw[0]
    scan = pos
    while scan < n:
        # skip whitespace mismatch: if first char is space, handle separately
        if text[scan] != first:
            scan += 1
            continue
        end = scan + len(raw)
        if end <= n and text[scan:end] == raw:
            return scan, end, end
        scan += 1

    return None, None, pos


def viet_words_with_offsets_robust(text: str):
    """
    Return list[(token, start, end)] where token keeps underthesea format (with '_').
    Offsets are computed by robust pointer scanning (no text.find(raw,cur)).
    """
    text = nfc(text)
    tokens = _tokenize_vi_words(text)

    out = []
    pos = 0
    for tok in tokens:
        s, e, pos2 = _scan_match_token(text, tok, pos)
        if s is None:
            # nếu không match được token này, bỏ qua để tránh lệch dây chuyền
            continue
        out.append((tok, s, e))
        pos = pos2
    return out


def word_bio_tags_robust(text: str, spans: List[Tuple[int,int,str]]):
    """
    Convert char-level spans -> word-level BIO tags using robust word offsets.
    Begin rule: B- nếu word_start đúng là span.start (cùng label), else I-.
    """
    text = nfc(text)
    spans = [(int(s), int(e), lab) for s, e, lab in spans]
    mask = build_char_mask(text, spans, prefer="first")
    begin_set = {(max(0, min(int(s), len(text))), lab) for (s, e, lab) in spans}

    words_info = viet_words_with_offsets_robust(text)  # [(tok, ws, we)]
    tags = []
    for (tok, ws, we) in words_info:
        lab = None
        for i in range(ws, min(we, len(mask))):
            if mask[i] is not None:
                lab = mask[i]
                break

        if lab is None:
            tags.append("O")
        else:
            tag = f"B-{lab}" if ((ws, lab) in begin_set) else f"I-{lab}"
            tags.append(tag)

    return words_info, tags


def encode_words_and_labels(tokenizer, tokens: List[str], word_tags: List[str], label2id: Dict[str,int], max_length=512):
    """
    Tokenize per segmented token (PhoBERT likes wordpieces). labels are expanded:
    - first subword keeps B-/I-
    - later subwords: B- becomes I-
    """
    input_ids = []
    labels = []

    for tok, tag in zip(tokens, word_tags):
        ids = tokenizer.encode(tok, add_special_tokens=False)
        if not ids:
            continue

        if tag == "O":
            input_ids.extend(ids)
            labels.extend([label2id["O"]] * len(ids))
        else:
            input_ids.append(ids[0])
            labels.append(label2id[tag])

            itag = ("I-" + tag[2:]) if tag.startswith("B-") else tag
            input_ids.extend(ids[1:])
            labels.extend([label2id[itag]] * (len(ids) - 1))

        if len(input_ids) >= (max_length - 2):
            input_ids = input_ids[: (max_length - 2)]
            labels = labels[: (max_length - 2)]
            break

    input_ids = tokenizer.build_inputs_with_special_tokens(input_ids)
    labels = [-100] + labels + [-100]
    attention_mask = [1] * len(input_ids)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def align_spans_to_token_labels_phobert(tokenizer, label2id: Dict[str,int], examples: Dict[str,Any], max_length=512):
    """
    Robust PhoBERT alignment:
    - word segmentation -> robust offsets via pointer scan
    - char spans -> word BIO -> subword labels
    """
    out = {"input_ids": [], "attention_mask": [], "labels": []}

    for text, starts, ends, labs in zip(examples["text"], examples["spans_start"], examples["spans_end"], examples["spans_label"]):
        text = nfc(text)
        spans = [(int(s), int(e), lab) for s, e, lab in zip(starts, ends, labs)]

        words_info, word_tags = word_bio_tags_robust(text, spans)
        tokens = [w for (w, _, _) in words_info]

        enc = encode_words_and_labels(tokenizer, tokens, word_tags, label2id, max_length=max_length)
        out["input_ids"].append(enc["input_ids"])
        out["attention_mask"].append(enc["attention_mask"])
        out["labels"].append(enc["labels"])

    return out

# Sanity check
def sanity_check_alignment(batch_enc: Dict[str,Any], n=5):
    for i in range(min(n, len(batch_enc["input_ids"]))):
        a = len(batch_enc["input_ids"][i])
        b = len(batch_enc["labels"][i])
        c = len(batch_enc["attention_mask"][i]) if "attention_mask" in batch_enc else a
        assert a == b == c, f"Length mismatch at sample {i}: ids={a}, labels={b}, mask={c}"
    return True

def has_offset_mapping(tokenizer) -> bool:
    # Fast tokenizer thường có thuộc tính is_fast = True
    return bool(getattr(tokenizer, "is_fast", False))

### Hàm đánh giá cho NER (BIO tagging)

- **`build_metrics_fn(id2label)`**: Tạo hàm `compute_metrics` dùng cho HuggingFace `Trainer`.
- **Cách hoạt động**:
  - Lấy **logits** và **labels**, dự đoán nhãn bằng `argmax`.
  - **Bỏ qua token bị mask** (`label = -100`, thường là special/subword).
  - Chuyển **id → nhãn BIO** bằng `id2label`.
  - Tính **Precision / Recall / F1** theo chuẩn NER (entity-level) với `seqeval`.
- **Kết quả**: Trả về dict metric để theo dõi chất lượng mô hình trong quá trình huấn luyện và đánh giá.

In [10]:
def build_metrics_fn(id2label: Dict[int, str]):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        true_preds, true_labels = [], []
        for p, y in zip(preds, labels):
            p_seq, y_seq = [], []
            for pi, yi in zip(p, y):
                if yi == -100:
                    continue
                p_seq.append(id2label[int(pi)])
                y_seq.append(id2label[int(yi)])
            true_preds.append(p_seq)
            true_labels.append(y_seq)

        return {
            "precision": precision_score(true_labels, true_preds),
            "recall": recall_score(true_labels, true_preds),
            "f1": f1_score(true_labels, true_preds),
        }
    return compute_metrics

# **Data preparation**

### Dữ liệu mẫu NER (JSONL) để test pipeline

- Khởi tạo **tập dữ liệu nhỏ minh hoạ** cho bài toán NER tiếng Việt.
- Mỗi mẫu gồm:
  - `text`: câu tiếng Việt
  - `labels`: danh sách span `[start, end, ENTITY]` ở mức ký tự
- Các loại thực thể ví dụ:
  - `PER`: Person
  - `LOC`: Location
  - `ORG`: Organization
  - `SUBJECT`: Môn học
- Ghi cùng một tập mẫu ra **train / valid / test** để:
  - kiểm tra nhanh pipeline
  - debug căn chỉnh span → token
  - sanity check huấn luyện mô hình

In [ ]:
# os.makedirs("data", exist_ok=True)

# sample_train = [
#     {"text":"Nguyễn Văn A hôm nay học môn Triết học","labels":[[0,12,"PER"],[29,38,"SUBJECT"]]},
#     {"text":"Trần Thị B sinh ra tại Hải Dương","labels":[[0,11,"PER"],[24,32,"LOC"]]},
#     {"text":"Lê Văn C đang học môn Toán rời rạc tại Đại học Khoa học Tự nhiên","labels":[[0,8,"PER"],[23,36,"SUBJECT"],[40,64,"ORG"]]},
#     {"text":"Phạm Thị D làm việc tại Thành phố Hồ Chí Minh","labels":[[0,10,"PER"],[24,45,"LOC"]]},
#     {"text":"Giảng viên Trần Văn F công tác tại Đại học Bách Khoa","labels":[[11,22,"PER"],[35,52,"ORG"]]},
# ]

# save_jsonl(sample_train, "data/train.jsonl")
# save_jsonl(sample_train, "data/valid.jsonl")
# save_jsonl(sample_train, "data/test.jsonl")

# print("Wrote sample data to data/*.jsonl")

### Chia dữ liệu JSONL thành train / valid / test

- **`split_jsonl(...)`**: Trộn ngẫu nhiên dữ liệu từ `data.jsonl` và chia thành 3 tập:
  - **Train**: 80%
  - **Validation**: 10%
  - **Test**: 10%
- **`seed=1`**: Cố định hạt giống ngẫu nhiên để **kết quả chia có thể tái lập**.
- **Output**:
  - `data/train.jsonl`
  - `data/valid.jsonl`
  - `data/test.jsonl`

Dùng để chuẩn bị dữ liệu huấn luyện và đánh giá mô hình NER.

In [ ]:
split_jsonl(
    input_path="/content/drive/MyDrive/medical_ner/data/data_old_method.jsonl",
    out_train="data/train.jsonl",
    out_valid="data/valid.jsonl",
    out_test="data/test.jsonl",
    train_ratio=0.8,
    valid_ratio=0.1,
    test_ratio=0.1,
    seed=1,
)

Split: train=2411, valid=301, test=302


### Chuẩn bị dữ liệu & DatasetDict cho huấn luyện NER

- **`set_seed(42)`**: Cố định seed để đảm bảo tính **tái lập** (shuffle, train).
- **Đọc & kiểm tra dữ liệu**:
  - `read_jsonl_and_rename_label`: chuẩn hoá khóa `label → labels`.
  - `validate_rows(..., strict=True, drop_invalid=True)`:  
    kiểm tra span, NFC, overlap; **loại bỏ mẫu lỗi** và log chi tiết.
- **Xây dựng nhãn BIO**:
  - `build_bio_label_maps(train + valid + test)` → `LABELS`, `label2id`, `id2label`.
- **Tạo HuggingFace `DatasetDict`**:
  - Chuyển mỗi split sang schema span-level chuẩn bằng `to_dataset`.
  - Gồm các split: `train`, `validation`, `test`.

Kết quả: Dataset sạch, nhãn BIO nhất quán, sẵn sàng cho bước tokenize & train NER.

In [13]:
set_seed(42)

train_rows = validate_rows(read_jsonl_and_rename_label("data/train.jsonl"), strict=True, drop_invalid=True, verbose=True)
valid_rows = validate_rows(read_jsonl_and_rename_label("data/valid.jsonl"), strict=True, drop_invalid=True, verbose=True)
test_rows  = validate_rows(read_jsonl_and_rename_label("data/test.jsonl"), strict=True, drop_invalid=True, verbose=True)

print(f"Loaded: train={len(train_rows)}, valid={len(valid_rows)}, test={len(test_rows)}")

LABELS, label2id, id2label = build_bio_label_maps(train_rows + valid_rows + test_rows)
print("BIO LABELS:", LABELS)

ds = DatasetDict({
    "train": to_dataset(train_rows),
    "validation": to_dataset(valid_rows),
    "test": to_dataset(test_rows),
})

Validation done | total=2411 | kept=2411 | removed=0
Validation done | total=301 | kept=301 | removed=0
[DROP] Row 122: Row 122 label end out of range: [53, 61, 'PLANT_PART'], len(text)=53
Validation done | total=302 | kept=301 | removed=1
Loaded: train=2411, valid=301, test=301
BIO LABELS: ['O', 'B-DISEASE', 'I-DISEASE', 'B-DOSAGE', 'I-DOSAGE', 'B-HERB', 'I-HERB', 'B-HUMAN_PART', 'I-HUMAN_PART', 'B-PLANT_PART', 'I-PLANT_PART', 'B-SYMPTOM', 'I-SYMPTOM']


### Preview

In [14]:
print(ds['test'][0])

{'text': 'Phía trong các bó libe gỗ là một khối mô mềm gọi là tủy, tủy có thể phát triển nhiều hay ít và đôi khi hoá mô cứng vài loại cây có ruột rỗng vì tủy bị tiêu hủy.', 'spans_start': [15, 38, 52, 57, 107, 131, 144], 'spans_end': [25, 44, 55, 60, 114, 135, 147], 'spans_label': ['PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART']}


# **Training**

### Tokenize + căn chỉnh nhãn span → token labels theo từng loại tokenizer

- **Chọn model/tokenizer** (XLM-R / PhoBERT / DeBERTa / …) và khởi tạo:
  - `AutoTokenizer.from_pretrained(model_name, use_fast=True)`
  - Kiểm tra `use_offsets = has_offset_mapping(tokenizer)` để biết tokenizer có hỗ trợ `offset_mapping` hay không.

- **Nhánh 1 — Có offsets (fast tokenizer)**:
  - Áp dụng cho các model như **XLM-R / DeBERTa** (và các tokenizer fast tương tự).
  - `ds.map(align_spans_to_token_labels, batched=True)`:
    - Tokenize văn bản và lấy offset từng token.
    - Căn nhãn BIO cho token dựa trên span char-level.

- **Nhánh 2 — Không có offsets (PhoBERT)**:
  - Áp dụng cho **PhoBERT** (thường không dùng offset_mapping ổn định theo cách này).
  - `ds.map(align_spans_to_token_labels_phobert, batched=True)`:
    - Tách từ tiếng Việt (word segmentation) + suy ra offsets theo scan.
    - Chuyển span → word BIO → subword labels (có `max_length`).

- **Kết quả**:
  - `ds_tok`: DatasetDict đã có các trường phục vụ train NER như `input_ids`, `attention_mask`, `labels` (và các field gốc nếu còn giữ).
  - In ra schema để kiểm tra: `print(ds_tok)` và `ds_tok["train"][0].keys()`.

In [15]:
MODEL_XLMR = "xlm-roberta-base"
MODEL_PHOBERT = "vinai/phobert-base"
MODEL_PHOBERT_LARGE = "vinai/phobert-large"
MODEL_MICROSOFT_NER = "microsoft/deberta-v3-large"

# model_name = MODEL_PHOBERT
# model_name = MODEL_XLMR
# model_name = MODEL_PHOBERT_LARGE
model_name = MODEL_MICROSOFT_NER

# tokenizer dùng để TRAIN + pad
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
use_offsets = has_offset_mapping(tokenizer)
print("TRAIN MODEL:", model_name)
print("Tokenizer class:", type(tokenizer))
print("Has offset_mapping:", use_offsets)

if use_offsets:
    # XLM-R / GGPT-NER
    ds_tok = ds.map(
        lambda ex: align_spans_to_token_labels(tokenizer, label2id, ex),
        batched=True,
        desc=f"Tokenize + Align spans→token labels (offsets) [{model_name}]",
    )
else:
    # PhoBERT
    ds_tok = ds.map(
        lambda ex: align_spans_to_token_labels_phobert(tokenizer, label2id, ex, max_length=256),
        batched=True,
        desc=f"Tokenize + Align spans→token labels (wordseg) [{model_name}]",
    )

print(ds_tok)
print(ds_tok["train"][0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

Tokenize + Align spans→token labels (offsets) [microsoft/deberta-v3-large]:   0%|          | 0/2411 [00:00<?, …

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokenize + Align spans→token labels (offsets) [microsoft/deberta-v3-large]:   0%|          | 0/301 [00:00<?, ?…

Tokenize + Align spans→token labels (offsets) [microsoft/deberta-v3-large]:   0%|          | 0/301 [00:00<?, ?…

DatasetDict({
    train: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2411
    })
    validation: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
})
dict_keys(['text', 'spans_start', 'spans_end', 'spans_label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])


### Sanity check độ dài input & nhãn

- Lấy một mẫu từ tập **train** đã tokenize và căn nhãn.
- In ra độ dài:
  - `input_ids`
  - `labels`
- **Assert** đảm bảo:
  ```text
  len(input_ids) == len(labels)
  ```
→ mỗi token đều có nhãn tương ứng (hoặc -100 cho token bỏ qua).

In [16]:
ex = ds_tok["train"][0]
print(len(ex["input_ids"]), len(ex["labels"]))
assert len(ex["input_ids"]) == len(ex["labels"])

72 72


### Debug căn chỉnh nhãn cho 1 mẫu (span → token BIO)

- Lấy 1 mẫu từ `ds["train"][0]`:
  - In **TEXT**
  - In danh sách **SPANS** dạng `(start, end, label)` để đối chiếu trực quan.

#### Nếu tokenizer có `offset_mapping` (fast tokenizer: XLM-R/DeBERTa…)
- Tokenize kèm `return_offsets_mapping=True` để lấy:
  - `tokens`: chuỗi token/subword
  - `offsets`: (start,end) trên `sample_text`
- Chạy lại `align_spans_to_token_labels` cho đúng mẫu đó để lấy `label_ids`.
- In theo từng token:
  - token, offset, đoạn text tương ứng `sample_text[s:e]`, nhãn BIO (`id2label`)  
  - `-100` được hiển thị thành **IGN** (bỏ qua, thường special token)

#### Nếu tokenizer không có offsets (PhoBERT)
- Dùng pipeline PhoBERT:
  1) `word_bio_tags_robust`: tách từ + suy offsets + gán nhãn BIO ở mức word  
  2) `encode_words_and_labels`: đổi word → subword và **lan truyền nhãn** sang subword
- In từng token/subword cùng nhãn (`IGN` nếu `-100`)

Mục đích: kiểm tra trực tiếp xem span char-level có được gán đúng lên token/subword hay không, phát hiện sớm lỗi lệch offset / sai B-I / sai vùng span.

In [17]:
# Debug 1 sample
sample_text  = ds["train"][0]["text"]
starts = ds["train"][0]["spans_start"]
ends   = ds["train"][0]["spans_end"]
labs   = ds["train"][0]["spans_label"]

print("TEXT:", sample_text)
print("SPANS:", list(zip(starts, ends, labs)))
print("-" * 80)

if has_offset_mapping(tokenizer):
    # XLM-R (or any Fast tokenizer with offsets)
    enc_dbg = tokenizer(sample_text, return_offsets_mapping=True, truncation=True, max_length=512)
    offsets = enc_dbg["offset_mapping"]
    tokens  = tokenizer.convert_ids_to_tokens(enc_dbg["input_ids"])

    one = align_spans_to_token_labels(
        tokenizer, label2id,
        {"text":[sample_text], "spans_start":[starts], "spans_end":[ends], "spans_label":[labs]}
    )
    label_ids = one["labels"][0]

    for tok, (s, e), lid in zip(tokens, offsets, label_ids):
        lab = "IGN" if lid == -100 else id2label[lid]
        piece = "" if (s == e) else sample_text[s:e]
        print(f"{tok:18s}  offset=({s:3d},{e:3d})  text='{piece}'  label={lab}")

else:
    # PhoBERT
    # 1) Word seg + word tags
    words_info, word_tags = word_bio_tags_robust(sample_text, list(zip(starts, ends, labs)))
    words = [w for (w, _, _) in words_info]

    # 2) Encode words -> subwords + labels
    enc = encode_words_and_labels(tokenizer, words, word_tags, label2id, max_length=512)

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    label_ids = enc["labels"]

    for tok, lid in zip(tokens, label_ids):
        lab = "IGN" if lid == -100 else id2label[int(lid)]
        print(f"{tok:18s}  label={lab}")

TEXT: Về cơ quan sinh sản Căn cứ vào 2 dạng chính là bào tử và hạt để chia thành 2 nhóm là thực vật sinh sản bằng bào tử hay bằng hạt.
SPANS: [(3, 19, 'PLANT_PART'), (47, 53, 'PLANT_PART'), (57, 60, 'PLANT_PART'), (108, 114, 'PLANT_PART'), (124, 127, 'PLANT_PART')]
--------------------------------------------------------------------------------
[CLS]               offset=(  0,  0)  text=''  label=IGN
▁V                  offset=(  0,  1)  text='V'  label=O
ề                   offset=(  1,  2)  text='ề'  label=O
▁c                  offset=(  2,  4)  text=' c'  label=I-PLANT_PART
ơ                   offset=(  4,  5)  text='ơ'  label=I-PLANT_PART
▁quan               offset=(  5, 10)  text=' quan'  label=I-PLANT_PART
▁sin                offset=( 10, 14)  text=' sin'  label=I-PLANT_PART
h                   offset=( 14, 15)  text='h'  label=I-PLANT_PART
▁s                  offset=( 15, 17)  text=' s'  label=I-PLANT_PART
ả                   offset=( 17, 18)  text='ả'  label=I-PLANT_PART
n     

## Khởi tạo mô hình Token Classification cho NER

- **`AutoModelForTokenClassification.from_pretrained(...)`**:
  - Tải backbone đã pretrain (`model_name`).
  - Gắn **head phân loại token** với số nhãn `num_labels = len(LABELS)`.

- **Ánh xạ nhãn**:
  - `label2id`: dùng trong huấn luyện (nhãn → chỉ số).
  - `id2label`: dùng khi suy luận & đánh giá (chỉ số → nhãn BIO).

- **Ý nghĩa**:
  - Mô hình sẵn sàng cho bài toán **NER/BIO tagging**.
  - Đảm bảo output logits khớp đúng **không gian nhãn BIO** đã xây dựng.

In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Huấn luyện mô hình NER (BIO tagging) với nhiều backbone khác nhau

- **Quản lý output theo mô hình**  
  Sử dụng bảng ánh xạ `MODEL → output_dir` để mỗi backbone (XLM-R, PhoBERT, PhoBERT-large, DeBERTa v3)  
  được lưu **checkpoint, log và best model riêng biệt**, tránh ghi đè và thuận tiện so sánh.

- **Cấu hình huấn luyện thống nhất**  
  Một bộ `TrainingArguments` dùng chung cho tất cả mô hình:
  - Đánh giá & lưu model theo **epoch**
  - Theo dõi **F1-score** và tự động load **best checkpoint**
  - Learning rate, batch size, weight decay được giữ cố định → **fair comparison**
  - `fp16=True` để tăng tốc và tiết kiệm VRAM

- **Data collator cho NER**  
  - Padding động theo batch
  - Token padding được gán nhãn `-100` để **không ảnh hưởng loss**

- **Trainer**  
  - Huấn luyện trên tập `train`, đánh giá trên `validation`
  - Dùng chung pipeline **span → token BIO alignment**
  - Metric đánh giá: **Precision / Recall / F1 (entity-level)**

Cách tổ chức này giúp **benchmark nhiều backbone NER** trong cùng một pipeline,
dễ mở rộng, dễ so sánh, và đảm bảo tính tái lập của thí nghiệm.

In [ ]:
OUTPUT_DIR_MAP = {
    MODEL_XLMR: "ner_xlmr",
    MODEL_PHOBERT: "ner_phobert",
    MODEL_PHOBERT_LARGE: "ner_phobert_large",
    MODEL_MICROSOFT_NER: "ner_deberta_v3_large",
}

args = TrainingArguments(
    output_dir=OUTPUT_DIR_MAP[model_name],
    eval_strategy="epoch",
    logging_strategy="steps",
    save_strategy="epoch",
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=True,
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=build_metrics_fn(id2label),
)

### Sanity check dữ liệu trước khi huấn luyện NER

- **Kiểm tra độ dài token–nhãn**  
  Với mỗi mẫu trong `train` và `validation`:
  - `len(input_ids) == len(labels)`  
  → đảm bảo **không lệch căn chỉnh** sau bước tokenize & gán nhãn.

- **Kiểm tra miền giá trị nhãn**  
  - Bỏ qua `-100` (token padding / special token).
  - Các nhãn còn lại phải nằm trong `[0, NUM_LABELS)`  
  → khớp với hệ nhãn BIO đã xây dựng.

- **Kiểm tra vocab tokenizer–model**  
  - Lấy `max(input_id)` trong tập train.
  - Đảm bảo `< model.config.vocab_size`  
  → tránh lỗi **tokenizer/model mismatch** (rất dễ gặp khi đổi backbone).

Các bước này giúp phát hiện sớm lỗi **lệch nhãn, sai chỉ số nhãn hoặc sai tokenizer**
trước khi bắt đầu huấn luyện, tiết kiệm thời gian debug.

In [20]:
NUM_LABELS = len(LABELS)
def sanity_check(split, name, n=50):
    for i in range(min(n, len(split))):
        x = split[i]["input_ids"]
        y = split[i]["labels"]
        assert len(x) == len(y), (name, i, len(x), len(y))
        for v in y:
            if v == -100:
                continue
            assert 0 <= int(v) < NUM_LABELS, (name, i, v)
    print("OK:", name)

sanity_check(ds_tok["train"], "train")
sanity_check(ds_tok["validation"], "validation")

# vocab check
mx = max(max(row["input_ids"]) for row in ds_tok["train"])
print("max input_id:", mx, "| vocab_size:", model.config.vocab_size)
assert mx < model.config.vocab_size, "input_ids vượt vocab => tokenizer/model mismatch"

OK: train
OK: validation
max input_id: 127982 | vocab_size: 128100


### Training

In [21]:
trains_stats = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.391800,0.339200,0.568456,0.622794,0.594386
2,0.300900,0.286277,0.637037,0.695588,0.665026
3,0.218300,0.257075,0.662425,0.727206,0.693305
4,0.154600,0.291739,0.678001,0.718382,0.697608
5,0.117800,0.295728,0.678238,0.747059,0.710987


### Lưu best checkpoint (model + tokenizer + label maps) lên Google Drive

- **Thiết lập thư mục gốc**: lưu toàn bộ checkpoint vào  
  `BASE_CKPT_DIR = /content/drive/MyDrive/checkpoint`.

- **Đặt tên run theo backbone** (`MODEL2RUN`)  
  Mỗi mô hình (XLM-R / PhoBERT / PhoBERT-large / DeBERTa v3) có **subfolder riêng** để:
  - không ghi đè checkpoint
  - dễ so sánh kết quả giữa các backbone

- **`safe_slug(model_name)`**  
  Tạo tên thư mục an toàn nếu model chưa có trong map  
  (vd: `microsoft/deberta-v3-large` → `microsoft_deberta-v3-large`).

- **Lưu artefacts cần thiết** vào `best_dir`:
  - `trainer.save_model(best_dir)`: lưu **model weights + config**
  - `tokenizer.save_pretrained(best_dir)`: lưu **tokenizer files**
  - `label_maps.json`: lưu **LABELS / label2id / id2label** để suy luận đúng BIO schema

- **Output**: in đường dẫn thư mục đã lưu (`Saved to: ...`).

In [ ]:
# Base checkpoint directory (Google Drive)
BASE_CKPT_DIR = "/content/drive/MyDrive/medical_ner/checkpoint"

# Map model → subdir
MODEL2RUN = {
    MODEL_XLMR: "ner_xlmr",
    MODEL_PHOBERT: "ner_phobert",
    MODEL_PHOBERT_LARGE: "ner_phobert_large",
    MODEL_MICROSOFT_NER: "ner_deberta_v3_large",
}

def safe_slug(s: str) -> str:
    # fallback: "microsoft/deberta-v3-large" -> "microsoft_deberta-v3-large"
    return (s or "unknown").replace("/", "_").replace(" ", "_")

run_name = MODEL2RUN.get(model_name, f"ner_{safe_slug(model_name)}")

# Final best checkpoint dir
best_dir = os.path.join(BASE_CKPT_DIR, run_name)
os.makedirs(best_dir, exist_ok=True)

trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

with open(os.path.join(best_dir, "label_maps.json"), "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "label2id": label2id, "id2label": id2label},
              f, ensure_ascii=False, indent=2)

print("Saved to:", best_dir)

Saved to: /content/drive/MyDrive/checkpoint/ner_deberta_v3_large


# **Evaluate**

### Các hàm tiện ích

- **Mục tiêu**: Load model từ `checkpoint_path`, chạy **evaluate + predict** trên tập test, in metric tổng quan,
  xuất **báo cáo theo từng entity**, và (tuỳ chọn) ghi file JSONL dự đoán dạng span kèm độ tin cậy.

##### 1) Tiện ích chung
- `softmax_np`: tính softmax numpy để lấy xác suất dự đoán theo lớp.
- `nfc`: chuẩn hoá Unicode NFC cho tiếng Việt.
- `has_offset_mapping`: phân biệt tokenizer **fast** (có offsets) và **PhoBERT-style** (không offsets).

##### 2) Tính metric (seqeval)
- `make_compute_metrics(id2label)`: chuyển `pred_id/label_id` → nhãn BIO, bỏ `-100`,
  tính **Precision/Recall/F1**.
- `show_overall_metrics`: in gọn các metric chính trên tập test.
- `make_entity_df`: tạo bảng **per-entity** (Precision/Recall/F1/Support) từ `classification_report`.

##### 3) Tokenize + align nhãn theo loại tokenizer
- **Fast tokenizer**: `align_spans_to_token_labels` dùng `offset_mapping` để gán BIO cho từng token.
- **PhoBERT**: pipeline `viet_words_with_offsets → word_bio_tags → encode_words_and_labels`
  để map span char-level → word BIO → subword labels.
- `build_ds_tok_for_model`: wrapper chọn đúng nhánh (fast vs phobert) và tạo `ds_tok`.

##### 4) Chuyển dự đoán token BIO → span (char-level) + confidence
- `tags_to_spans_with_score`: gom chuỗi BIO thành span `[start,end,type,avg_conf]`.
- **Fast**: `build_pred_spans_fast` dùng offsets của tokenizer để dựng span.
- **PhoBERT**: `build_pred_spans_phobert` gom dự đoán theo word_ids rồi map về offsets của word.

##### 5) Xuất JSONL dự đoán (compact & unified)
- `export_predictions_to_jsonl_compact_unified`:
  - Ghi mỗi dòng gồm `id`, `text`, `label` (GT), `Comments` (nếu có), `pred` (span dự đoán).
  - Có `min_pred` để lọc span có confidence thấp.

##### 6) Hàm chính: `evaluate_from_checkpoint(...)`
Thực hiện lần lượt:
1. Load tokenizer từ `checkpoint_path` và build `ds_tok` tương ứng.
2. Load `AutoModelForTokenClassification` với `label2id/id2label`.
3. Tạo `Trainer` chỉ để evaluate/predict + compute_metrics.
4. `evaluate()` → in metric tổng quan.
5. `predict()` → lấy logits, preds, xác suất token.
6. Tạo báo cáo per-entity (DataFrame) để xem entity nào mạnh/yếu.
7. (Tuỳ chọn) export JSONL dự đoán theo span + confidence.

Kết quả trả về: `(test_metrics, df_report)` và (nếu bật) file JSONL dự đoán để phân tích/visualize sau.

In [ ]:
# Utils
def softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def nfc(text: str) -> str:
    import unicodedata
    return unicodedata.normalize("NFC", text or "")

def has_offset_mapping(tokenizer) -> bool:
    return bool(getattr(tokenizer, "is_fast", False))

# Metrics helpers
def make_compute_metrics(id2label):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        true_labels, true_preds = [], []
        for p_row, l_row in zip(preds, labels):
            sent_true, sent_pred = [], []
            for p, l in zip(p_row, l_row):
                if l == -100:
                    continue
                sent_true.append(id2label[int(l)])
                sent_pred.append(id2label[int(p)])
            true_labels.append(sent_true)
            true_preds.append(sent_pred)

        return {
            "precision": precision_score(true_labels, true_preds),
            "recall": recall_score(true_labels, true_preds),
            "f1": f1_score(true_labels, true_preds),
        }
    return compute_metrics

def show_overall_metrics(metrics: dict):
    # Trainer.evaluate thường trả key eval_loss + (compute_metrics -> eval_precision/recall/f1)
    keys = ["eval_precision", "eval_recall", "eval_f1", "eval_loss"]
    print("\nOVERALL TEST METRICS")
    for k in keys:
        if k in metrics:
            print(f"{k.replace('eval_', '').upper():10s}: {metrics[k]:.4f}")

def make_entity_df(true_labels, true_preds):
    rep = classification_report(
        true_labels,
        true_preds,
        output_dict=True,
        zero_division=0,
    )

    rows = []
    for label, scores in rep.items():
        if label in ("micro avg", "macro avg", "weighted avg"):
            continue
        rows.append({
            "Entity": label,
            "Precision": float(scores["precision"]),
            "Recall": float(scores["recall"]),
            "F1": float(scores["f1-score"]),
            "Support": int(scores["support"]),
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    return df.sort_values(["Support", "F1"], ascending=[False, False]).reset_index(drop=True)

# Align spans -> token labels (FAST tokenizer: offsets)
def align_spans_to_token_labels(tokenizer, label2id, examples):
    texts = examples["text"]
    starts_batch = examples["spans_start"]
    ends_batch   = examples["spans_end"]
    labs_batch   = examples["spans_label"]

    enc = tokenizer(
        texts,
        truncation=True,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    all_labels = []
    for text, starts, ends, labs, offsets in zip(
        texts, starts_batch, ends_batch, labs_batch, enc["offset_mapping"]
    ):
        text = text or ""
        char_mask = [None] * len(text)
        spans = list(zip(starts, ends, labs))

        for s, e, lab in spans:
            s = max(0, min(int(s), len(text)))
            e = max(0, min(int(e), len(text)))
            for i in range(s, e):
                char_mask[i] = lab

        token_labels = []
        for (start, end) in offsets:
            if start == end:
                token_labels.append(-100)
                continue

            lab = None
            for i in range(start, min(end, len(char_mask))):
                if char_mask[i] is not None:
                    lab = char_mask[i]
                    break

            if lab is None:
                token_labels.append(label2id["O"])
            else:
                # "Begin" nếu token start == span.start của cùng label
                is_begin = any((int(s) == start and l == lab) for s, e, l in spans)
                tag = f"B-{lab}" if is_begin else f"I-{lab}"
                token_labels.append(label2id[tag])

        all_labels.append(token_labels)

    enc.pop("offset_mapping")
    enc["labels"] = all_labels
    return enc

# PhoBERT (SLOW tokenizer): wordseg pipeline (no offsets)
def viet_words_with_offsets(text: str):
    text = nfc(text)
    words = text.split()
    out = []
    cur = 0
    for w in words:
        pos = text.find(w, cur)
        if pos == -1:
            continue
        s = pos
        e = pos + len(w)
        out.append((w, s, e))
        cur = e
    return out

def build_char_mask(text: str, spans):
    mask = [None] * len(text)
    for s, e, lab in spans:
        s = max(0, min(int(s), len(text)))
        e = max(0, min(int(e), len(text)))
        for i in range(s, e):
            mask[i] = lab
    return mask

def word_bio_tags(text: str, spans):
    text = nfc(text)
    words_info = viet_words_with_offsets(text)
    mask = build_char_mask(text, spans)

    tags = []
    for (w, ws, we) in words_info:
        lab = None
        for i in range(ws, min(we, len(mask))):
            if mask[i] is not None:
                lab = mask[i]
                break

        if lab is None:
            tags.append("O")
        else:
            is_begin = (ws < len(mask) and mask[ws] == lab and (ws == 0 or mask[ws-1] != lab))
            tags.append(f"B-{lab}" if is_begin else f"I-{lab}")
    return words_info, tags

def encode_words_and_labels(tok, words, word_tags, label2id, max_length=256):
    input_ids = []
    labels = []

    for w, tag in zip(words, word_tags):
        ids = tok.encode(w, add_special_tokens=False)
        if len(ids) == 0:
            continue

        if tag == "O":
            input_ids.extend(ids)
            labels.extend([label2id["O"]] * len(ids))
        else:
            input_ids.append(ids[0])
            labels.append(label2id[tag])

            itag = ("I-" + tag[2:]) if tag.startswith("B-") else tag
            input_ids.extend(ids[1:])
            labels.extend([label2id[itag]] * (len(ids) - 1))

        if len(input_ids) >= (max_length - 2):
            input_ids = input_ids[: (max_length - 2)]
            labels = labels[: (max_length - 2)]
            break

    input_ids = tok.build_inputs_with_special_tokens(input_ids)
    labels = [-100] + labels + [-100]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

def align_spans_to_token_labels_phobert(tokenizer, label2id, examples, max_length=256):
    out = {"input_ids": [], "attention_mask": [], "labels": []}

    for text, starts, ends, labs in zip(
        examples["text"], examples["spans_start"], examples["spans_end"], examples["spans_label"]
    ):
        text = nfc(text)
        spans = [(s, e, l) for s, e, l in zip(starts, ends, labs)]

        words_info, word_tags = word_bio_tags(text, spans)
        words = [w for (w, _, _) in words_info]

        enc = encode_words_and_labels(tokenizer, words, word_tags, label2id, max_length=max_length)
        out["input_ids"].append(enc["input_ids"])
        out["attention_mask"].append(enc["attention_mask"])
        out["labels"].append(enc["labels"])

    return out

# Build tokenized dataset for either fast or PhoBERT
def build_ds_tok_for_model(ds, tokenizer, label2id, model_name, max_length_phobert=256):
    use_offsets = has_offset_mapping(tokenizer)
    print("EVAL MODEL:", model_name)
    print("Tokenizer class:", type(tokenizer))
    print("Has offset_mapping:", use_offsets)

    if use_offsets:
        ds_tok = ds.map(
            lambda ex: align_spans_to_token_labels(tokenizer, label2id, ex),
            batched=True,
            desc=f"Tokenize + Align spans→token labels (offsets) [{model_name}]",
        )
    else:
        ds_tok = ds.map(
            lambda ex: align_spans_to_token_labels_phobert(
                tokenizer, label2id, ex, max_length=max_length_phobert
            ),
            batched=True,
            desc=f"Tokenize + Align spans→token labels (wordseg) [{model_name}]",
        )
    return ds_tok

# Pred spans builder
def tags_to_spans_with_score(offsets, tags, confs):
    spans = []
    cur_type, cur_s, cur_e = None, None, None
    cur_scores = []

    def flush():
        nonlocal cur_type, cur_s, cur_e, cur_scores
        if cur_type is not None and cur_s is not None and cur_e is not None and cur_e > cur_s:
            spans.append([int(cur_s), int(cur_e), cur_type, float(np.mean(cur_scores)) if cur_scores else 0.0])
        cur_type, cur_s, cur_e, cur_scores = None, None, None, []

    for (s, e), tag, sc in zip(offsets, tags, confs):
        if s is None or e is None or s == e:
            continue

        if tag is None or tag == "O":
            flush()
            continue

        if tag.startswith("B-"):
            flush()
            cur_type = tag[2:]
            cur_s, cur_e = s, e
            cur_scores = [float(sc)]
        elif tag.startswith("I-"):
            typ = tag[2:]
            if cur_type != typ or cur_type is None:
                flush()
                cur_type = typ
                cur_s, cur_e = s, e
                cur_scores = [float(sc)]
            else:
                cur_e = max(cur_e, e)
                cur_scores.append(float(sc))
        else:
            flush()

    flush()
    return spans

def build_pred_spans_fast(text, pred_ids, pred_conf, tok, id2label, max_len_tokens=None):
    enc = tok(
        text,
        truncation=True,
        max_length=max_len_tokens,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )
    offsets = enc["offset_mapping"]
    tags = [id2label[int(x)] for x in pred_ids]
    confs = [float(x) for x in pred_conf]
    L = min(len(offsets), len(tags), len(confs))
    return tags_to_spans_with_score(offsets[:L], tags[:L], confs[:L])

def encode_words_and_labels_with_map(tok, words, max_length=256):
    input_ids = []
    word_ids = []

    for wi, w in enumerate(words):
        ids = tok.encode(w, add_special_tokens=False)
        if len(ids) == 0:
            continue
        for tid in ids:
            input_ids.append(tid)
            word_ids.append(wi)
        if len(input_ids) >= (max_length - 2):
            input_ids = input_ids[: (max_length - 2)]
            word_ids = word_ids[: (max_length - 2)]
            break

    input_ids = tok.build_inputs_with_special_tokens(input_ids)
    word_ids = [None] + word_ids + [None]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "word_ids": word_ids}

def word_tags_from_token_preds(word_ids, pred_ids, pred_conf, id2label):
    tmp = {}
    for wi, pid, cf in zip(word_ids, pred_ids, pred_conf):
        if wi is None:
            continue
        tmp.setdefault(wi, []).append((id2label[int(pid)], float(cf)))

    max_wi = max(tmp.keys()) if tmp else -1
    word_tags, word_confs = [], []
    for wi in range(max_wi + 1):
        arr = tmp.get(wi, [])
        if not arr:
            word_tags.append("O")
            word_confs.append(0.0)
            continue
        word_tags.append(arr[0][0])
        word_confs.append(float(np.mean([x[1] for x in arr])))
    return word_tags, word_confs

def build_pred_spans_phobert(text, pred_ids, pred_conf, tok, id2label, max_len_tokens=256):
    text = nfc(text)
    words_info = viet_words_with_offsets(text)
    words = [w for (w, _, _) in words_info]

    enc = encode_words_and_labels_with_map(tok, words, max_length=max_len_tokens)
    word_ids = enc["word_ids"]

    L = min(len(word_ids), len(pred_ids), len(pred_conf))
    word_ids  = word_ids[:L]
    pred_ids  = pred_ids[:L]
    pred_conf = pred_conf[:L]

    word_tags, word_confs = word_tags_from_token_preds(word_ids, pred_ids, pred_conf, id2label)

    spans = []
    cur_type, cur_s, cur_e = None, None, None
    cur_scores = []

    def flush():
        nonlocal cur_type, cur_s, cur_e, cur_scores
        if cur_type is not None and cur_s is not None and cur_e is not None and cur_e > cur_s:
            spans.append([int(cur_s), int(cur_e), cur_type, float(np.mean(cur_scores)) if cur_scores else 0.0])
        cur_type, cur_s, cur_e, cur_scores = None, None, None, []

    for (w, ws, we), tag, cf in zip(words_info, word_tags, word_confs):
        if tag is None or tag == "O":
            flush()
            continue
        if tag.startswith("B-"):
            flush()
            cur_type = tag[2:]
            cur_s, cur_e = ws, we
            cur_scores = [float(cf)]
        elif tag.startswith("I-"):
            typ = tag[2:]
            if cur_type != typ or cur_type is None:
                flush()
                cur_type = typ
                cur_s, cur_e = ws, we
                cur_scores = [float(cf)]
            else:
                cur_e = max(cur_e, we)
                cur_scores.append(float(cf))
        else:
            flush()

    flush()
    return spans

# Unified Exporter
def export_predictions_to_jsonl_compact_unified(
    raw_test,
    ds_tok_test,
    preds,
    pred_prob,
    tok,
    id2label,
    out_path="pred_compact.jsonl",
    min_pred=0.0,
):
    fast = has_offset_mapping(tok)

    with open(out_path, "w", encoding="utf-8") as f:
        for i in range(len(raw_test)):
            raw_item = raw_test[i]
            text = raw_item.get("text", "")
            sample_id = raw_item.get("id", i)
            gt_label = raw_item.get("label", [])
            comments = raw_item.get("Comments", [])

            pred_ids_i = preds[i].tolist()
            conf_i     = pred_prob[i].tolist()

            # ép đúng số token đã dùng khi predict
            max_len_tokens = len(ds_tok_test[i]["input_ids"])

            if fast:
                pred_spans = build_pred_spans_fast(
                    text, pred_ids_i, conf_i, tok, id2label, max_len_tokens=max_len_tokens
                )
            else:
                pred_spans = build_pred_spans_phobert(
                    text, pred_ids_i, conf_i, tok, id2label, max_len_tokens=max_len_tokens
                )

            if min_pred is not None and float(min_pred) > 0:
                pred_spans = [sp for sp in pred_spans if float(sp[3]) >= float(min_pred)]

            rec = {
                "id": sample_id,
                "text": text,
                "label": gt_label,
                "Comments": comments,
                "pred": pred_spans,  # [[s,e,TYPE,avg_conf], ...]
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# MAIN: Evaluate + Export
def evaluate_from_checkpoint(
    ds,
    checkpoint_path,
    label2id,
    id2label,
    max_length_phobert=256,
    save_jsonl_path=None,
    min_pred=0.0,
):
    # 1) tokenizer + tokenize dataset
    tok = AutoTokenizer.from_pretrained(checkpoint_path)  # không ép use_fast
    ds_tok_ckpt = build_ds_tok_for_model(
        ds, tok, label2id, model_name=checkpoint_path, max_length_phobert=max_length_phobert
    )

    # 2) model
    model = AutoModelForTokenClassification.from_pretrained(
        checkpoint_path,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )

    # 3) collator (pad labels = -100)
    data_collator = DataCollatorForTokenClassification(tokenizer=tok)

    # 4) trainer
    trainer_eval = Trainer(
        model=model,
        tokenizer=tok,  # warning deprecate nhưng vẫn chạy
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(id2label),
    )

    # 5) evaluate overall
    test_metrics = trainer_eval.evaluate(ds_tok_ckpt["test"])
    show_overall_metrics(test_metrics)

    # 6) predict
    preds_output = trainer_eval.predict(ds_tok_ckpt["test"])
    logits = preds_output.predictions          # (N, L, C)
    labels = preds_output.label_ids            # (N, L)
    preds  = np.argmax(logits, axis=-1)        # (N, L)

    p_all = softmax_np(logits, axis=-1)        # (N, L, C)
    pred_prob = np.take_along_axis(p_all, preds[..., None], axis=-1).squeeze(-1)  # (N, L)

    # 7) per-entity report
    true_labels, true_preds = [], []
    for p_row, l_row in zip(preds, labels):
        sent_true, sent_pred = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            sent_true.append(id2label[int(l)])
            sent_pred.append(id2label[int(p)])
        true_labels.append(sent_true)
        true_preds.append(sent_pred)

    df_report = make_entity_df(true_labels, true_preds)
    display(df_report)

    # 8) export JSONL
    if save_jsonl_path is not None:
        export_predictions_to_jsonl_compact_unified(
            raw_test=ds["test"],
            ds_tok_test=ds_tok_ckpt["test"],
            preds=preds,
            pred_prob=pred_prob,
            tok=tok,
            id2label=id2label,
            out_path=save_jsonl_path,
            min_pred=float(min_pred),
        )
        print(f"[Saved] predictions JSONL -> {save_jsonl_path} (min_pred={min_pred})")

    return test_metrics, df_report

### Chọn checkpoint để đánh giá & xuất dự đoán

- **Chọn model đã train** bằng `checkpoint_path` (PhoBERT / PhoBERT-large / XLM-R / DeBERTa v3).
  Ví dụ đang dùng:
  - `.../checkpoint/ner_deberta_v3_large`

- **Tự nhận diện PhoBERT để set tham số riêng**
  - `is_phobert = ("phobert" in checkpoint_path.lower())`
  - Nếu là PhoBERT: cần `max_length_phobert` (do pipeline wordseg/subword có giới hạn độ dài)
  - Nếu không phải PhoBERT (XLM-R/DeBERTa): không cần tham số đặc thù, vẫn có thể giữ `256` như mặc định.

- **`evaluate_from_checkpoint(...)` sẽ làm 3 việc chính**
  1) **Evaluate** trên tập `test` và in metric tổng quan (**Precision/Recall/F1/Loss**)  
  2) **Predict** và tạo bảng **per-entity report** (`df_report`) để xem entity nào mạnh/yếu  
  3) **Xuất JSONL dự đoán** ra `pred_compact.jsonl` (span `[start,end,TYPE,avg_conf]`)
     - `min_pred=0`: không lọc theo confidence (giữ tất cả span)

- **Output nhận được**
  - `test_metrics`: dict metric từ `Trainer.evaluate/predict`
  - `df_report`: DataFrame báo cáo theo từng entity (Precision/Recall/F1/Support)
  - file `pred_compact.jsonl` để phân tích/visualize về sau

In [ ]:
# --- Pick checkpoint ---
# checkpoint_path = "/content/drive/MyDrive/checkpoint/ner_phobert"
# checkpoint_path = "/content/drive/MyDrive/checkpoint/ner_phobert_large"
# checkpoint_path = "/content/drive/MyDrive/checkpoint/ner_xlmr"
checkpoint_path = "/content/drive/MyDrive/medical_ner/checkpoint/ner_deberta_v3_large"

# --- Auto params (PhoBERT only) ---
is_phobert = ("phobert" in checkpoint_path.lower())
max_length_phobert = 256 if is_phobert else None  # or keep 256, but None makes intent clear

test_metrics, df_report = evaluate_from_checkpoint(
    ds,
    checkpoint_path=checkpoint_path,
    label2id=label2id,
    id2label=id2label,
    max_length_phobert=max_length_phobert if max_length_phobert is not None else 256,
    save_jsonl_path="pred_compact.jsonl",
    min_pred=0,
)

The tokenizer you are loading from '/content/drive/MyDrive/checkpoint/ner_deberta_v3_large' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


EVAL MODEL: /content/drive/MyDrive/checkpoint/ner_deberta_v3_large
Tokenizer class: <class 'transformers.models.deberta_v2.tokenization_deberta_v2_fast.DebertaV2TokenizerFast'>
Has offset_mapping: True


Tokenize + Align spans→token labels (offsets) [/content/drive/MyDrive/checkpoint/ner_deberta_v3_large]:   0%| …

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokenize + Align spans→token labels (offsets) [/content/drive/MyDrive/checkpoint/ner_deberta_v3_large]:   0%| …

...

 3


wandb: You chose "Don't visualize my results"



OVERALL TEST METRICS
PRECISION : 0.6919
RECALL    : 0.7355
F1        : 0.7131
LOSS      : 0.3265


,Entity,Precision,Recall,F1,Support
0,PLANT_PART,0.771574,0.795812,0.783505,382
1,HERB,0.769841,0.808333,0.788618,360
2,DISEASE,0.609524,0.612440,0.610979,209
3,SYMPTOM,0.515837,0.581633,0.546763,196
4,HUMAN_PART,0.560000,0.675862,0.612500,145
5,DOSAGE,0.945205,0.945205,0.945205,73


[Saved] predictions JSONL -> pred_compact.jsonl (min_pred=0)


# **Inference**

### Chuyển dự đoán BIO → danh sách thực thể (char-level spans)

- **`bio_to_spans(text, offsets, pred_ids, id2label)`**:
  - Nhận kết quả dự đoán ở mức **token BIO** và **offsets** của tokenizer.
  - Ghép các token liên tiếp thành **thực thể hoàn chỉnh** ở mức ký tự.

- **Cách hoạt động**:
  - Duyệt lần lượt từng token `(start, end)` và nhãn dự đoán.
  - Bỏ qua token rỗng (`start == end`) và xử lý nhãn `O` (kết thúc thực thể đang mở).
  - Quy tắc ghép:
    - `B-<TYPE>` → bắt đầu thực thể mới.
    - `I-<TYPE>` → nối tiếp thực thể cùng loại.
    - Nếu gặp `I-<TYPE>` nhưng **khác TYPE hiện tại** → tự động tách và mở thực thể mới.
  - Kết thúc vòng lặp thì flush thực thể còn lại (nếu có).

- **Output**:
  - Danh sách dict dạng:
    ```json
    {
      "start": 10,
      "end": 25,
      "label": "ORG",
      "text": "Đại học Bách Khoa"
    }
    ```
  - Tọa độ **char-level** khớp trực tiếp với `text` gốc.

In [25]:
def bio_to_spans(text: str, offsets: List[List[int]], pred_ids: List[int], id2label: Dict[int, str]):
    entities = []
    cur = None  # [start, end, TYPE]

    for (s, e), pid in zip(offsets, pred_ids):
        if s == e:
            continue

        tag = id2label[int(pid)]
        if tag == "O":
            if cur:
                entities.append(cur)
                cur = None
            continue

        pref, lab = tag.split("-", 1)

        if pref == "B" or (cur and cur[2] != lab):
            if cur:
                entities.append(cur)
            cur = [s, e, lab]
        else:
            if cur:
                cur[1] = e
            else:
                cur = [s, e, lab]

    if cur:
        entities.append(cur)

    return [{"start": s, "end": e, "label": lab, "text": text[s:e]} for s, e, lab in entities]

### Suy luận NER từ checkpoint (tự động hỗ trợ Fast tokenizer & PhoBERT) và trả về spans

- **Load artefacts từ `best_dir`**:
  - `AutoTokenizer`, `AutoModelForTokenClassification`, và `label_maps.json`
  - Chuẩn hoá `id2label` về đúng kiểu **int → str** để map nhãn BIO chính xác.

- **Tự phát hiện tokenizer có offsets hay không**
  - `tokenizer_has_offsets(tok)` thử gọi `return_offsets_mapping=True`
  - Nếu **có offsets** → chạy nhánh Fast tokenizer (XLM-R/DeBERTa…)
  - Nếu **không có offsets** → chạy nhánh PhoBERT (word-seg + map subword → word)

### Nhánh 1 — Fast tokenizer (có `offset_mapping`)
- Encode `text` với offsets → predict `pred_ids` → `bio_to_spans(...)`
- Output: danh sách thực thể dạng:
  `{"start":..., "end":..., "label":..., "text":...}` (tọa độ char-level)

### Nhánh 2 — PhoBERT (không offsets)
- Word segmentation + offsets từ `viet_words_with_offsets`
- Encode giống lúc train (dummy tag `O`) để tạo `input_ids`
- Predict subword → lấy tag của **subword đầu tiên mỗi word** (`subword_preds_to_word_tags`)
- Ghép lại thành span theo offsets word (`wordtags_to_spans`)

- **`predict_entities(text)`**: API thống nhất cho cả 2 loại tokenizer, trả về spans cuối cùng.

> Lưu ý: câu demo có nội dung sức khoẻ; đoạn code chỉ dùng để **trích xuất thực thể** (NER),
> không phải chẩn đoán hay kết luận y khoa.

In [ ]:
# Load best model/tokenizer
tok = AutoTokenizer.from_pretrained(best_dir, use_fast=True)  # nếu model là PhoBERT thì vẫn sẽ load slow tokenizer
mdl = AutoModelForTokenClassification.from_pretrained(best_dir)
mdl.eval()

with open(os.path.join(best_dir, "label_maps.json"), "r", encoding="utf-8") as f:
    maps = json.load(f)

# ensure id2label keys are int
id2label_loaded = maps["id2label"]
if isinstance(next(iter(id2label_loaded.keys())), str):
    id2label_loaded = {int(k): v for k, v in id2label_loaded.items()}

label2id_loaded = maps.get("label2id", None)
LABELS_LOADED = maps.get("labels", None)

def tokenizer_has_offsets(tokenizer) -> bool:
    """True nếu tokenizer hỗ trợ return_offsets_mapping (fast tokenizer)."""
    try:
        _ = tokenizer("test", return_offsets_mapping=True)
        return True
    except Exception:
        return False

# Helpers for PhoBERT inference (no offsets)
def subword_preds_to_word_tags(words, word_tags_gt, pred_ids, input_ids, tokenizer, id2label):
    """
    Map dự đoán subword -> word-level tag bằng cách:
    - tokenize từng word để biết số subword của word đó
    - lấy tag của subword đầu tiên làm tag word (giống cách bạn gán B/I xuống subword lúc train)
    """
    # pred_ids tương ứng với input_ids có special tokens
    # bỏ <s> ... </s>
    pred_core = pred_ids[1:-1]
    ids_core  = input_ids[1:-1]

    # Tính số subword cho mỗi word (đúng theo encode_words_and_labels)
    lens = []
    for w in words:
        ids = tokenizer.encode(w, add_special_tokens=False)
        if len(ids) == 0:
            lens.append(0)
        else:
            lens.append(len(ids))

    # Map: mỗi word lấy pred của subword đầu tiên
    word_pred_tags = []
    ptr = 0
    for L in lens:
        if L <= 0:
            # word bị skip vì encode ra rỗng -> dự đoán O
            word_pred_tags.append("O")
            continue

        if ptr >= len(pred_core):
            word_pred_tags.append("O")
            continue

        tag0 = id2label[int(pred_core[ptr])]
        word_pred_tags.append(tag0)

        ptr += L

    # Đảm bảo độ dài khớp
    if len(word_pred_tags) != len(words):
        word_pred_tags = word_pred_tags[:len(words)] + ["O"] * max(0, len(words) - len(word_pred_tags))

    return word_pred_tags


def wordtags_to_spans(text, words_info, word_tags):
    """
    words_info: [(w, ws, we), ...] offsets theo text gốc
    word_tags:  ["B-XXX"/"I-XXX"/"O", ...]
    -> list dict spans
    """
    spans = []
    cur = None  # [start,end,label]

    for (w, s, e), tag in zip(words_info, word_tags):
        if tag == "O" or tag == "IGN":
            if cur:
                spans.append(cur)
                cur = None
            continue

        pref, lab = tag.split("-", 1)

        if pref == "B" or (cur and cur[2] != lab):
            if cur:
                spans.append(cur)
            cur = [s, e, lab]
        else:
            # I
            if cur:
                cur[1] = e
            else:
                cur = [s, e, lab]

    if cur:
        spans.append(cur)

    return [{"start": s, "end": e, "label": lab, "text": text[s:e]} for s, e, lab in spans]

def predict_entities(text: str, max_length: int = 256):
    text = nfc(text)

    if tokenizer_has_offsets(tok):
        # FAST tokenizer (XLM-R)
        enc = tok(
            text,
            return_offsets_mapping=True,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
        )
        offsets = enc.pop("offset_mapping")[0].tolist()

        with torch.no_grad():
            out = mdl(**enc)

        pred_ids = out.logits.argmax(-1)[0].tolist()
        return bio_to_spans(text, offsets, pred_ids, id2label_loaded)

    else:
        # PhoBERT
        # 1) word segmentation + offsets
        words_info = viet_words_with_offsets(text)  # [(w, ws, we), ...]
        words = [w for (w, _, _) in words_info]

        # 2) encode giống train (nhãn tạm O)
        #    labels chỉ để đủ format; inference không cần labels.
        dummy_tags = ["O"] * len(words)
        enc = encode_words_and_labels(tok, words, dummy_tags, label2id_loaded, max_length=max_length)

        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]

        # 3) predict subword
        inputs = {
            "input_ids": torch.tensor([input_ids], dtype=torch.long),
            "attention_mask": torch.tensor([attention_mask], dtype=torch.long),
        }
        with torch.no_grad():
            out = mdl(**inputs)

        pred_ids = out.logits.argmax(-1)[0].tolist()

        # 4) subword -> word tag (lấy tag subword đầu tiên mỗi word)
        word_pred_tags = subword_preds_to_word_tags(
            words=words,
            word_tags_gt=dummy_tags,
            pred_ids=pred_ids,
            input_ids=input_ids,
            tokenizer=tok,
            id2label=id2label_loaded
        )

        # 5) word tags -> spans theo offsets word
        return wordtags_to_spans(text, words_info, word_pred_tags)

# Demo
print(predict_entities("Hôm nay tôi bị đau đầu và ù tai có thể đã bị si đa."))

The tokenizer you are loading from '/content/drive/MyDrive/checkpoint/ner_deberta_v3_large' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[{'start': 14, 'end': 22, 'label': 'SYMPTOM', 'text': ' đau đầu'}, {'start': 26, 'end': 31, 'label': 'SYMPTOM', 'text': 'ù tai'}, {'start': 44, 'end': 50, 'label': 'DISEASE', 'text': ' si đa'}]
